# PHASE 2 — Prétraitement et Feature Engineering
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Transformer les données brutes en format exploitable pour les modèles.

**Jours 8-12 du plan**

---
### Contenu :
- Jour 8 : Nettoyage (suppression >90% NaN, doublons)
- Jour 9 : Imputation (médiane numérique, mode catégorielle)
- Jour 10 : Encodage sans fuite de données (Frequency + OneHot)
- Jour 11 : Split stratifié + StandardScaler (SMOTE réservé Phase 4)
- Jour 12 : Feature engineering mobile money

In [ ]:
# === IMPORT ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')

In [ ]:
# === CHARGEMENT (si dataset déjà téléchargé) ===
# Si vous venez de la Phase 1, le dataset est déjà en mémoire.
# Sinon, exécuter les cellules de téléchargement du notebook 01_EDA.

# Vérifier si df existe
try:
    print(f'Dataset déjà chargé : {df.shape}')
except NameError:
    print('Dataset non chargé. Exécutez d\'abord les cellules de téléchargement du notebook 01_EDA.')

---
## JOUR 8 — Nettoyage

**Objectif :** Supprimer les colonnes >90% de NaN et les doublons. Garder une trace des colonnes supprimées (justification mémoire).

In [ ]:
# === Suppression des colonnes >90% NaN ===
missing = (df.isnull().sum() / len(df) * 100)
cols_to_drop = missing[missing > 90].index.tolist()

print(f'=== Colonnes supprimées (>90% NaN) : {len(cols_to_drop)} ===')
for c in cols_to_drop:
    print(f'  - {c} : {missing[c]:.1f}%')

df_clean = df.drop(columns=cols_to_drop)
print(f'\nDimensions avant : {df.shape}')
print(f'Dimensions après : {df_clean.shape}')

In [ ]:
# === Suppression des doublons ===
dup = df_clean.duplicated().sum()
if dup > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'{dup} doublons supprimés')
else:
    print('Aucun doublon trouvé')

In [ ]:
# === Séparation features / target ===
if 'isFraud' in df_clean.columns:
    y = df_clean['isFraud']
    X = df_clean.drop(columns=['isFraud', 'TransactionID', 'TransactionDT_dt'], errors='ignore')
    print(f'X : {X.shape}, y : {y.shape}')
    print(f'Taux de fraude : {y.mean()*100:.2f}%')
else:
    print('ERREUR : isFraud non trouvé')

In [ ]:
# === Séparation numériques / catégorielles ===
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Variables numériques : {len(num_cols)}')
print(f'Variables catégorielles : {len(cat_cols)}')
print(f'\nCatégorielles : {cat_cols[:20]}...' if len(cat_cols) > 20 else f'\nCatégorielles : {cat_cols}')

**Justification mémoire (section 3.3.2) :**
La suppression des colonnes à >90% de NaN est justifiée par l'impossibilité d'imputer des variables quasi-vides sans introduire un bruit excessif ou un biais significatif. Les colonnes supprimées concernent principalement les données d'identité (appareil, navigateur) qui ne sont pas disponibles pour toutes les transactions.

---
## JOUR 9 — Imputation

**Objectif :** Imputer les valeurs manquantes — médiane pour les numériques, mode pour les catégorielles.

In [ ]:
# === Vérification des NaN restants ===
total_nan = X[num_cols + cat_cols].isnull().sum().sum()
print(f'Total NaN avant imputation : {total_nan}')
print(f'% de NaN : {total_nan / (X.shape[0] * X.shape[1]) * 100:.2f}%')

In [ ]:
# === Imputation ===
# NOTE : L'imputation sera intégrée dans le ColumnTransformer après le split.
# On prépare ici les transformateurs.

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

print('Imputers prêts :')
print(f'  - Numérique : médiane')
print(f'  - Catégorielle : mode')

**Commentaire mémoire :** La médiane est préférée à la moyenne pour les variables numériques car elle est robuste aux valeurs extrêmes (outliers fréquents dans les données de transaction). Le mode est utilisé pour les variables catégorielles, avec une catégorie 'manquant' si nécessaire.

---
## JOUR 10 — Split + Encodage (sans fuite de données)

**Règle d'or :** Split AVANT tout encodage ou scaling. Les fréquences pour le Frequency Encoding sont calculées UNIQUEMENT sur X_train.

In [ ]:
# === SPLIT STRATIFIÉ 80/20 ===
X_train, X_test, y_train, y_test = train_test_split(
    X[num_cols + cat_cols], y, test_size=0.2, stratify=y, random_state=42
)

print(f'X_train : {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Fraude Train : {y_train.mean()*100:.2f}%')
print(f'Fraude Test : {y_test.mean()*100:.2f}%')
print('\n✅ Split effectué AVANT encodage — aucune fuite de données')

In [ ]:
# === FREQUENCY ENCODING (haute cardinalité) ===
# Calcul des fréquences UNIQUEMENT sur X_train

# Identifier les colonnes à haute cardinalité (>10 catégories uniques)
high_card_cols = []
low_card_cols = []
for col in cat_cols:
    n_unique = X_train[col].nunique()
    if n_unique > 10:
        high_card_cols.append(col)
    else:
        low_card_cols.append(col)

print(f'Colonnes à haute cardinalité (>10) : {len(high_card_cols)}')
print(f'Colonnes à faible cardinalité : {len(low_card_cols)}')

In [ ]:
# Application du Frequency Encoding
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()
freq_maps = {}  # pour sauvegarde

for col in high_card_cols:
    # Fréquence sur TRAIN uniquement
    freq = X_train[col].value_counts() / len(X_train)
    freq_maps[col] = freq
    
    # Appliquer au train et test
    X_train_enc[col] = X_train[col].map(freq)
    X_test_enc[col] = X_test[col].map(freq)
    
    # Catégories absentes du train → fréquence 0
    X_test_enc[col] = X_test_enc[col].fillna(0)

print(f'✅ Frequency Encoding appliqué sur {len(high_card_cols)} colonnes')
print('  (fréquences calculées uniquement sur X_train)')

In [ ]:
# === OneHotEncoding (faible cardinalité) ===
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

if len(low_card_cols) > 0:
    train_ohe = ohe.fit_transform(X_train[low_card_cols])
    test_ohe = ohe.transform(X_test[low_card_cols])
    
    # Récupérer les noms des colonnes encodées
    ohe_cols = ohe.get_feature_names_out(low_card_cols)
    
    # Convertir en DataFrame
    train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_enc.index)
    test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test_enc.index)
    
    # Supprimer les colonnes originales et ajouter les encodées
    X_train_enc = X_train_enc.drop(columns=low_card_cols)
    X_test_enc = X_test_enc.drop(columns=low_card_cols)
    
    X_train_enc = pd.concat([X_train_enc, train_ohe_df], axis=1)
    X_test_enc = pd.concat([X_test_enc, test_ohe_df], axis=1)
    
    print(f'✅ OneHotEncoding : {len(low_card_cols)} colonnes → {len(ohe_cols)} indicatrices')
else:
    print('Aucune colonne à faible cardinalité à encoder')

In [ ]:
# === Alternative présentée dans le mémoire : Target Encoding ===
# Dans le mémoire (section 2.4.2), mentionner que le Target Encoding
# (leave-one-out ou CV) est une alternative au Frequency Encoding.
# Non utilisé ici pour éviter tout risque de leakage.

print('Discussion mémoire : Target Encoding (category_encoders.TargetEncoder)')
print('  - Avantage : capture la corrélation avec la cible')
print('  - Risque : fuite de données si mal implémenté')
print('  - Choix retenu : Frequency Encoding (plus sûr)')

---
## JOUR 11 — Normalisation (StandardScaler)

**Règle d'or :** Le scaler est ajusté sur X_train uniquement, puis appliqué à X_test.

In [ ]:
# === StandardScaler ===
# Identifier les colonnes numériques restantes (après encodage)
remaining_num_cols = [c for c in num_cols if c in X_train_enc.columns]

scaler = StandardScaler()
X_train_scaled = X_train_enc.copy()
X_test_scaled = X_test_enc.copy()

if len(remaining_num_cols) > 0:
    X_train_scaled[remaining_num_cols] = scaler.fit_transform(X_train_enc[remaining_num_cols])
    X_test_scaled[remaining_num_cols] = scaler.transform(X_test_enc[remaining_num_cols])
    print(f'✅ StandardScaler appliqué sur {len(remaining_num_cols)} variables numériques')
else:
    print('Aucune variable numérique à scaler')

In [ ]:
# === Vérification finale ===
print('=== Dimensions finales ===')
print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'X_test_scaled : {X_test_scaled.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_test : {y_test.shape}')

print(f'\n=== Valeurs manquantes ===')
print(f'Train : {X_train_scaled.isnull().sum().sum()} NaN')
print(f'Test : {X_test_scaled.isnull().sum().sum()} NaN')

In [ ]:
# === Sauvegarde du scaler ===
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(freq_maps, 'models/freq_maps.pkl')
if len(low_card_cols) > 0:
    joblib.dump(ohe, 'models/ohe_encoder.pkl')
print('✅ Scaler, freq_maps et OHE sauvegardés dans models/')

---
## JOUR 12 — Feature Engineering "Mobile Money"

**Objectif :** Créer des variables pertinentes pour le contexte togolais.

In [ ]:
# Recharger X propre pour feature engineering
X_fe = X.copy()

# 1. log_amount
X_fe['log_amount'] = np.log1p(X_fe['TransactionAmt'])
print('✅ log_amount = log(TransactionAmt + 1)')

In [ ]:
# 2. tx_count_by_card1 (fréquence cumulée par identifiant)
if 'card1' in X_fe.columns:
    X_fe['tx_count_by_card1'] = X_fe.groupby('card1')['TransactionAmt'].transform('count')
    print(f'✅ tx_count_by_card1 créé (min={X_fe["tx_count_by_card1"].min()}, '
          f'max={X_fe["tx_count_by_card1"].max()})')
else:
    print('⚠️ card1 non trouvé')

In [ ]:
# 3. time_since_last_tx_card1 (delta temporel — indicateur SIM swap)
if 'card1' in X_fe.columns and 'TransactionDT' in X_fe.columns:
    X_fe_sorted = X_fe.sort_values(['card1', 'TransactionDT'])
    X_fe['time_since_last_tx_card1'] = X_fe_sorted.groupby('card1')['TransactionDT'].diff()
    X_fe['time_since_last_tx_card1'] = X_fe['time_since_last_tx_card1'].fillna(0)
    print(f'✅ time_since_last_tx_card1 créé')
    print(f'  - Délai max : {X_fe["time_since_last_tx_card1"].max():.0f} secondes')
    print(f'  - Délai moyen : {X_fe["time_since_last_tx_card1"].mean():.0f} secondes')
else:
    print('⚠️ card1 ou TransactionDT non trouvé')

In [ ]:
# 4. hour et dayofweek (à refaire si perdu)
if 'TransactionDT' in X_fe.columns and 'TransactionDT_dt' not in X_fe.columns:
    import datetime
    start_date = datetime.datetime(2017, 12, 1)
    dt_col = X_fe['TransactionDT'].apply(lambda x: start_date + datetime.timedelta(seconds=x))
    X_fe['hour'] = dt_col.dt.hour
    X_fe['dayofweek'] = dt_col.dt.dayofweek
    print('✅ hour et dayofweek créés')

In [ ]:
# === Sauvegarde des données prétraitées ===
# Pour Google Colab, sauvegarder sur Drive
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = '/content/drive/MyDrive/FRAUDX/data/'
os.makedirs(save_path, exist_ok=True)

pd.DataFrame(X_train_scaled).to_pickle(f'{save_path}X_train.pkl')
pd.DataFrame(X_test_scaled).to_pickle(f'{save_path}X_test.pkl')
pd.DataFrame(y_train).to_pickle(f'{save_path}y_train.pkl')
pd.DataFrame(y_test).to_pickle(f'{save_path}y_test.pkl')
pd.DataFrame(X_fe).to_pickle(f'{save_path}X_featured.pkl')

print(f'✅ Données sauvegardées dans {save_path}')

---
## Tableau pour le mémoire — Correspondance IEEE-CIS ↔ Mobile Money Togo

| Variable IEEE-CIS | Équivalent conceptuel Togo | Justification |
|---|---|---|
| `TransactionAmt` | Montant recharge/transfert/paiement (FCFA) | Variable universelle de transaction |
| `card1`-`card6` | Identifiant SIM / device (IMEI) | Identifiant unique du terminal utilisé |
| `C1`-`C14` | Fréquence transactions agent/numéro | Comportement du compte sur une période |
| `D1`-`D15` | Délai depuis dernière transaction (SIM swap) | Détection de changement d'appareil |
| `tx_count_by_card1` | Fréquence d'utilisation SIM | Indicateur d'activité anormale |
| `time_since_last_tx_card1` | Indicateur SIM swap | Délai anormalement court entre transactions |
| `ProductCD` | Type d'opération (recharge/transfert/paiement) | Catégorisation des canaux mobile money |
| `hour` / `dayofweek` | Heure de pointe (8h-10h / 17h-19h) | Pics d'activité mobile money |

**Section mémoire :** Tableau 3.x présenté dans la section 3.3.3 "Discussion sur la transférabilité au contexte togolais".

---
## Récapitulatif Phase 2

- **Colonnes supprimées** : >90% NaN (quantité variable selon exécution)
- **Split** : 80/20 stratifié ✓
- **Imputation** : médiane (num) + mode (cat) — intégrée dans le pipeline
- **Encodage** : Frequency Encoding (haute cardinalité) + OneHot (faible cardinalité) — **sans fuite** ✓
- **Scaling** : StandardScaler ajusté sur train, appliqué à test ✓
- **SMOTE** : réservé à la Phase 4 (sur train uniquement) ⏳
- **Feature engineering** : log_amount, tx_count_by_card1, time_since_last_tx_card1, hour, dayofweek ✓
- **Sauvegarde** : données prétraitées dans Google Drive ✓